Exact transport/exchange reaction sets, complete and with recorded-ID exclusions.
Jaccard = shared/union; empty/empty = 1. These are structural comparisons.

Set the relative input/output constants in the first code cell. No local Python files
are imported.


In [ ]:
from pathlib import Path
from itertools import combinations
import pandas as pd

INPUT_DIR = Path("inputs")
RESULT_DIR = Path("outputs")
OUTPUT_DIR = RESULT_DIR / "transport_set_comparisons"
REFERENCE_PATH = INPUT_DIR / "haeconGEM1.xml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE = RESULT_DIR / "whole_model_enrichment_results"
inputs = [SOURCE / "model_reaction_membership.tsv", SOURCE / "background_reactions.tsv"]
m = pd.read_csv(inputs[0], sep="\t")
bg = pd.read_csv(inputs[1], sep="\t").set_index("reaction_id")
models = ["egg", "L1", "L2", "L3", "L4f", "L4m", "Am", "Af"]
sets = {s: set(g.reaction_id) for s, g in m.groupby("model")}
transport = bg.subsystem.str.contains("transport", case=False, regex=False)
boundary = bg.subsystem.eq("Exchange/demand reactions")
groups = {
    "Transport/exchange (annotation-defined)": set(bg.index[transport | boundary]),
    "Transport annotations (excluding exchange/demand category)": set(
        bg.index[transport]
    ),
}
for sub in sorted(bg.loc[transport | boundary, "subsystem"].unique()):
    groups["Subsystem: " + sub] = set(bg.index[bg.subsystem.eq(sub)])
primary = [
    ("egg", "L1"),
    ("L1", "L2"),
    ("L2", "L3"),
    ("L3", "L4union"),
    ("L4f", "L4m"),
    ("L4f", "Af"),
    ("L4m", "Am"),
    ("Af", "Am"),
    ("L3", "L4f"),
    ("L3", "L4m"),
]
pairs = list(combinations(models, 2))
for pair in primary:
    if pair not in pairs and pair[::-1] not in pairs:
        pairs.append(pair)
for pair in primary:
    if pair not in pairs:
        pairs.append(pair)
rows = []
members = []
for a, b in pairs:
    for group, G in groups.items():
        A = sets[a] & G
        B = sets[b] & G
        shared = A & B
        onlya = A - B
        onlyb = B - A
        union = A | B
        rows.append(
            dict(
                model_A=a,
                model_B=b,
                group=group,
                n_A=len(A),
                n_B=len(B),
                shared=len(shared),
                only_A=len(onlya),
                only_B=len(onlyb),
                union=len(union),
                symmetric_difference=len(onlya | onlyb),
                jaccard=len(shared) / len(union) if union else 1.0,
                different_sets=A != B,
                shared_IDs=";".join(sorted(shared)),
                only_A_IDs=";".join(sorted(onlya)),
                only_B_IDs=";".join(sorted(onlyb)),
            )
        )
        for rid in sorted(union):
            members.append(
                dict(
                    model_A=a,
                    model_B=b,
                    group=group,
                    reaction_id=rid,
                    name=bg.loc[rid, "name"],
                    subsystem=bg.loc[rid, "subsystem"],
                    membership=(
                        "shared"
                        if rid in shared
                        else "only_A" if rid in onlya else "only_B"
                    ),
                )
            )
d = pd.DataFrame(rows)
membership = pd.DataFrame(members)
d.to_csv(OUTPUT_DIR / "all_pairwise_sets.tsv", sep="\t", index=False)
membership.to_csv(OUTPUT_DIR / "reaction_membership_by_pair.tsv", sep="\t", index=False)
primary_rows = pd.concat(
    [d[(d.model_A == a) & (d.model_B == b)] for a, b in primary], ignore_index=True
)
primary_rows.to_csv(
    OUTPUT_DIR / "paper_pairs_all_transport_categories.tsv", sep="\t", index=False
)
summary = primary_rows[
    primary_rows.group.eq("Transport/exchange (annotation-defined)")
].drop(columns=["shared_IDs", "only_A_IDs", "only_B_IDs", "group"])
summary.to_csv(OUTPUT_DIR / "paper_pairs_transport_summary.tsv", sep="\t", index=False)
assert sets["L4union"] == sets["L4f"] | sets["L4m"]
assert (d.n_A == d.shared + d.only_A).all() and (d.n_B == d.shared + d.only_B).all()
assert (d.union == d.shared + d.only_A + d.only_B).all()
assert (d.symmetric_difference == d.only_A + d.only_B).all()


In [ ]:
import cobra

TRANSPORT_EXCLUDED = [
    "MAR00022",
    "MAR00023",
    "MAR00036",
    "MAR00565",
    "MAR00666",
    "MAR00693",
    "MAR00694",
    "MAR01455",
    "MAR01506",
    "MAR01507",
    "MAR01563",
    "MAR01564",
    "MAR01686",
    "MAR01688",
    "MAR01690",
    "MAR01698",
    "MAR01705",
    "MAR02960",
    "MAR02993",
    "MAR03031",
    "MAR03036",
    "MAR03041",
    "MAR03131",
    "MAR06005",
    "MAR06013",
    "MAR06020",
    "MAR06028",
    "MAR06035",
    "MAR06040",
    "MAR06044",
    "MAR06049",
    "MAR06051",
    "MAR06053",
    "MAR06629",
    "MAR06630",
    "MAR06631",
    "MAR06632",
    "MAR06633",
    "MAR06634",
    "MAR06635",
    "MAR06636",
    "MAR06637",
    "MAR06638",
    "MAR06639",
    "MAR06641",
    "MAR06642",
    "MAR06644",
    "MAR06646",
    "MAR06647",
    "MAR06648",
    "MAR06649",
    "MAR06651",
    "MAR06652",
    "MAR06653",
    "MAR06654",
    "MAR06655",
    "MAR06656",
    "MAR06657",
    "MAR06658",
    "MAR06659",
    "MAR06660",
    "MAR06662",
    "MAR06663",
    "MAR06664",
    "MAR06665",
    "MAR06666",
    "MAR06667",
    "MAR06668",
    "MAR06669",
    "MAR06670",
    "MAR06671",
    "MAR06672",
    "MAR06673",
    "MAR06674",
    "MAR06675",
    "MAR06676",
    "MAR06677",
    "MAR06678",
    "MAR06679",
    "MAR06680",
    "MAR06682",
    "MAR06685",
    "MAR06686",
    "MAR06687",
    "MAR06690",
    "MAR06691",
    "MAR06692",
    "MAR06693",
    "MAR06694",
    "MAR06695",
    "MAR06697",
    "MAR06699",
    "MAR06700",
    "MAR06701",
    "MAR06702",
    "MAR06703",
    "MAR06704",
    "MAR06705",
    "MAR08697",
    "MAR08698",
    "MAR08699",
    "MAR08700",
    "MAR08702",
    "MAR08703",
    "MAR08704",
    "MAR08706",
    "MAR08709",
    "MAR08710",
    "MAR08711",
    "MAR08712",
    "MAR08713",
    "MAR08714",
    "MAR08715",
    "MAR08716",
    "MAR08717",
    "MAR08718",
    "MAR08719",
    "MAR08721",
    "MAR08722",
    "MAR09147",
    "MAR09210",
    "MAR09404",
    "MAR09405",
    "MAR09573",
    "MAR09574",
    "MAR09656",
    "MAR09713",
    "MAR09903",
    "MAR10065",
    "MAR10483",
    "MAR10492",
    "MAR11894",
    "MAR11895",
    "MAR11896",
    "MAR11897",
    "MAR13067",
    "MAR20001",
    "MAR20003",
    "MAR20005",
    "MAR20006",
    "MAR20007",
    "MAR20008",
    "MAR20067",
    "MAR20068",
]
full = cobra.io.read_sbml_model(str(REFERENCE_PATH))
reasons = {rid: "recorded final reaction exclusion" for rid in TRANSPORT_EXCLUDED}
excluded = set(reasons)
pd.DataFrame(
    [dict(reaction_id=rid, reason=reason) for rid, reason in sorted(reasons.items())]
).to_csv(OUTPUT_DIR / "reaction_exclusion_catalog.tsv", sep="\t", index=False)
DEST = OUTPUT_DIR / "pairs_with_recorded_exclusions"
DEST.mkdir(exist_ok=True)
source = membership[
    membership.group.eq("Transport/exchange (annotation-defined)")
].copy()
removed = source[source.reaction_id.isin(excluded)].copy()
removed.to_csv(DEST / "excluded_reaction_members.tsv", sep="\t", index=False)
filtered = source[~source.reaction_id.isin(excluded)].copy()
summary_rows = []
for a, b in primary:
    g = filtered[(filtered.model_A == a) & (filtered.model_B == b)].copy()
    changed = g[g.membership.ne("shared")].copy()
    changed["present_in_A"] = changed.membership.eq("only_A")
    changed["present_in_B"] = changed.membership.eq("only_B")
    changed["unique_to_model"] = changed.membership.map({"only_A": a, "only_B": b})
    changed["reaction_equation"] = changed.reaction_id.map(
        lambda rid: full.reactions.get_by_id(rid).build_reaction_string(
            use_metabolite_names=True
        )
    )
    changed["name"] = changed.reaction_id.map(
        lambda rid: full.reactions.get_by_id(rid).name
    )
    changed = changed.drop(columns=["group"]).sort_values(
        ["membership", "subsystem", "reaction_id"]
    )
    changed.to_csv(DEST / f"{a}_vs_{b}_changed_transporters.tsv", sep="\t", index=False)
    g.drop(columns=["group"]).to_csv(
        DEST / f"{a}_vs_{b}_all_members.tsv", sep="\t", index=False
    )
    shared = int(g.membership.eq("shared").sum())
    oa = int(g.membership.eq("only_A").sum())
    ob = int(g.membership.eq("only_B").sum())
    u = len(g)
    summary_rows.append(
        dict(
            model_A=a,
            model_B=b,
            n_A=shared + oa,
            n_B=shared + ob,
            shared=shared,
            only_A=oa,
            only_B=ob,
            union=u,
            symmetric_difference=oa + ob,
            jaccard=shared / u if u else 1.0,
            different_sets=bool(oa + ob),
        )
    )
    assert len(changed) == oa + ob and (not set(changed.reaction_id) & excluded)
summary_filtered = pd.DataFrame(summary_rows)
summary_filtered.to_csv(
    OUTPUT_DIR / "paper_pairs_transport_summary_with_exclusions.tsv",
    sep="\t",
    index=False,
)
